# Helmholtz equation on a torus

Fourier-patch geometry and Helmholtz consistency benchmark.


In [ ]:
from pathlib import Path
import sys

project = Path.cwd().resolve()
if project.name == 'notebooks':
    project = project.parent
sys.path.insert(0, str(project))

import pysurfacefun as psf

output_dir = project / 'notebook_outputs'
output_dir.mkdir(exist_ok=True)

## Build the Torus

where `n` is the number of Chebyshev points per patch, and `nu`, `nv` are the patch counts in the two parameter directions.

In [ ]:
n = 9
nu = 16
nv = 32

dom = psf.torus(n=n, nu=nu, nv=nv)
print(f'number of patches = {dom.npatches}')
print(f'points per patch  = {dom.n} x {dom.n}')
print(f'surface area      = {psf.surfacearea(dom):.12f}')

## Geometry p-refinement Check

As `n` increases, the computed surface area should stabilize.

In [ ]:
for n in [5, 7, 9, 11]:
    dom_n = psf.torus(n=n, nu=nu, nv=nv)
    area = psf.surfacearea(dom_n)
    print(f'n = {n:2d}, patches = {dom_n.npatches:2d}, surface area = {area:.12f}')

## Define a Smooth Exact Function

For a simple consistency check, choose

$$u(x,y,z) = x+y+z.$$

Then form the right-hand side by applying the numerical surface Laplacian:

$$f = \Delta_\Gamma u + \alpha u.$$

Solving

$$(\Delta_\Gamma + \alpha I)u_h = f$$

should recover the original sampled function.

In [ ]:
n = 9
dom = psf.torus(n=n, nu=nu, nv=nv)

u_exact = psf.field(lambda x, y, z: x + y + z, dom)
alpha = 20.0
rhs = psf.lap(u_exact) + alpha * u_exact

## Solve the Helmholtz-type Problem

Here the operator is not rank deficient because of the positive zeroth-order term `alpha`.

In [ ]:
problem = psf.SurfaceProblem(dom, variables='u', namespace={'alpha': alpha, 'rhs': rhs})
problem.add_equation('lap(u) + alpha*u = rhs')
u_h = problem.solve()

relerr = psf.norm(u_h - u_exact, 'inf') / psf.norm(u_exact, 'inf')
print(f'relative L_inf error = {relerr:.3e}')

## p-refinement for the Solve

In [ ]:
for n in [7, 9, 11,19]:
    dom_n = psf.torus(n=n, nu=nu, nv=nv)
    exact_n = psf.field(lambda x, y, z: x + y + z, dom_n)
    rhs_n = psf.lap(exact_n) + alpha * exact_n
    problem_n = psf.SurfaceProblem(dom_n, variables='u', namespace={'alpha': alpha, 'rhs': rhs_n})
    problem_n.add_equation('lap(u) + alpha*u = rhs')
    sol_n = problem_n.solve()
    err_n = psf.norm(sol_n - exact_n, 'inf') / psf.norm(exact_n, 'inf')
    print(f'n = {n:2d}, relative L_inf error = {err_n:.3e}')

## Optional VTU Export

This requires `meshio`. Open the generated `.vtu` file in ParaView.

In [ ]:
try:
    psf.write_vtu(output_dir / 'torus_helmholtz_solution.vtu', u_h)
    print('wrote torus_helmholtz_solution.vtu')
except Exception as exc:
    print('VTU export skipped:', exc)